In [ ]:
"""
================================================================================
CRESTA MLE INTERVIEW — PRACTICAL CODING PROBLEM
================================================================================
DOMAIN:  LLM-as-Judge Auto-Scorer for Conversation Quality
TIME:    60-90 minutes
LEVEL:   Mid-Senior MLE

SCENARIO
--------
You are an MLE on Cresta's Conversation Intelligence team. Contact centers
currently score agent quality by having QA managers listen to ~2% of calls
and rate them on a rubric. This is slow, expensive, and statistically noisy.

Cresta's product auto-scores 100% of conversations using an LLM-as-judge.
This gives managers full visibility instead of a tiny sample, and lets
coaches give agents specific, data-backed feedback.

Your job: build the scoring pipeline, calibrate it against human raters,
and measure where the LLM judge agrees and disagrees with humans.

THE RUBRIC (5 criteria, scored 1-5 each):
------------------------------------------
1. GREETING        — Did the agent greet professionally and identify themselves?
2. PROBLEM_SOLVING — Did the agent understand and address the customer's issue?
3. EMPATHY         — Did the agent acknowledge the customer's feelings?
4. RESOLUTION      — Was a clear resolution or next step provided?
5. CLOSING         — Did the agent summarize and close professionally?

Each scored 1-5:
  1 = Missing/terrible
  2 = Poor
  3 = Adequate
  4 = Good
  5 = Excellent

YOUR TASK (4 parts)
-------------------
Part 1: Build the scoring prompt & rubric system         [15 min]
Part 2: Run LLM judge on conversations                  [15 min]
Part 3: Calibrate against human raters                   [20 min]
Part 4: Analyze disagreements & bias                     [15 min]
================================================================================
"""

import json
import re
import math
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter


# ─────────────────────────────────────────────────────────────────────────────
# CONVERSATION DATA — 8 scored conversations (do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Conversation:
    id: str
    transcript: str
    metadata: Dict  # team, duration, etc.


@dataclass
class HumanScore:
    conversation_id: str
    rater_id: str
    scores: Dict[str, int]  # criterion -> 1-5 score
    notes: str = ""


CRITERIA = ["greeting", "problem_solving", "empathy", "resolution", "closing"]

CONVERSATIONS = [
    Conversation("conv_001", """
[00:00] AGENT: Thank you for calling Meridian Wireless, my name is Sarah. How can I help you today?
[00:05] CUSTOMER: Hi Sarah, I was charged $47.99 for a premium data add-on but I never signed up for it.
[00:12] AGENT: Oh, I'm sorry to hear about that unexpected charge. That must be frustrating to see something you didn't authorize on your bill. Let me pull up your account right away.
[00:20] CUSTOMER: Yeah, it's annoying. Account is AC-7842961.
[00:25] AGENT: Thank you. I can see the charge here. It looks like this was added during a system migration last month — several customers were affected. I'm going to remove the add-on right now and issue a full refund of $47.99 to your next billing cycle.
[00:40] CUSTOMER: Okay, that works.
[00:42] AGENT: Great. So to confirm — I've removed the premium data add-on and you'll see a $47.99 credit on your next statement. I've also added a note to prevent this from happening again. Is there anything else I can help with?
[00:55] CUSTOMER: No, that's all. Thanks for sorting it out quickly.
[00:58] AGENT: You're welcome! Thank you for being a loyal customer. Have a great day.
""".strip(), {"team": "billing", "duration_sec": 58}),

    Conversation("conv_002", """
[00:00] AGENT: Meridian support.
[00:03] CUSTOMER: My internet has been dropping out every 20 minutes for three days. I work from home and this is killing me.
[00:10] AGENT: Account number?
[00:13] CUSTOMER: AC-3391054. Can you actually fix this? I've called twice already.
[00:18] AGENT: Let me look... yeah I see packet loss on your line. It's a hardware issue, I can't fix it remotely. I'll escalate to tier 2. Someone will call you within 24 hours.
[00:35] CUSTOMER: Twenty-four hours?! I need this fixed NOW. I can't work like this.
[00:40] AGENT: I understand but there's nothing more I can do from here. You'll get a call within 24 hours. Anything else?
[00:48] CUSTOMER: I guess not. This is really disappointing.
[00:51] AGENT: Sorry about that. Bye.
""".strip(), {"team": "technical", "duration_sec": 51}),

    Conversation("conv_003", """
[00:00] AGENT: Thank you for calling Meridian Wireless! I'm Lisa, how are you doing today?
[00:05] CUSTOMER: I want to cancel my account. I'm switching to Apex Mobile.
[00:10] AGENT: I'm sorry to hear you're thinking about leaving us. Before we go ahead, may I ask what's driving the switch? I want to make sure we explore every option.
[00:18] CUSTOMER: Your prices keep going up. I'm paying $89 and Apex offers $59 for the same thing.
[00:25] AGENT: I completely understand — nobody wants to feel like they're overpaying, especially when there's a competitive offer out there. You've been with us for three years and I really value that loyalty. Let me see what I can do.
[00:35] AGENT: I can offer you our loyalty rate of $62 per month, locked in for 12 months. That's very close to the Apex price, and you keep your current device plan and number with no disruption.
[00:48] CUSTOMER: $62? That's actually not bad. What's the catch?
[00:52] AGENT: No catch — it's a genuine loyalty rate for customers who've been with us over two years. The only thing to note is the 12-month commitment. If you cancel within that period, any remaining device installment balance comes due.
[01:02] CUSTOMER: Okay, I'll take it.
[01:05] AGENT: Wonderful! I've applied the $62 rate starting today. You'll see it on your next bill, and I'll send a confirmation email. Thank you so much for staying with us — is there anything else I can help with?
[01:15] CUSTOMER: No, that's it. Thanks Lisa.
[01:18] AGENT: My pleasure! Have a wonderful evening.
""".strip(), {"team": "retention", "duration_sec": 78}),

    Conversation("conv_004", """
[00:00] AGENT: Hello, Meridian Wireless, David speaking. What can I help you with?
[00:05] CUSTOMER: I have two things. I need to update my address and there's a weird charge on my bill.
[00:12] AGENT: Sure, happy to help with both. Let's start with the address. What's your account number and new address?
[00:18] CUSTOMER: AC-9912347. New address is 445 Birch Lane, Apartment 3B, Portland, OR 97201.
[00:28] AGENT: Got it, address updated. Now, since you have our home internet, we'll need to schedule a transfer. Our installations team will reach out within 48 hours. Now about the charge — what are you seeing?
[00:40] CUSTOMER: There's a $14.99 charge for Device Protection Plus. I never added that.
[00:48] AGENT: Let me check... it was added through the app on February 2nd. Might you have tapped something accidentally while browsing?
[00:58] CUSTOMER: Maybe, but I didn't mean to. Can you remove it?
[01:02] AGENT: Absolutely. I've removed it and refunded this month's $14.99. For the prior two months, I'll submit a dispute — you'll hear back in 5-7 business days. Your reference number is RF-44217.
[01:15] CUSTOMER: Okay, thanks David.
[01:18] AGENT: You're welcome. To recap: address is updated, internet transfer is in progress, Device Protection is removed with one month refunded and two months in dispute. Good luck with the move — anything else?
[01:30] CUSTOMER: That covers it.
[01:32] AGENT: Great, have a good one!
""".strip(), {"team": "billing", "duration_sec": 92}),

    Conversation("conv_005", """
[00:00] AGENT: Meridian Wireless, how can I help?
[00:03] CUSTOMER: When does my contract end? Account AC-2210038.
[00:08] AGENT: Your agreement ends August 15th, 2026. After that it's month-to-month.
[00:15] CUSTOMER: Perfect, thanks.
[00:17] AGENT: Happy to help, bye.
""".strip(), {"team": "general", "duration_sec": 17}),

    Conversation("conv_006", """
[00:00] AGENT: Thank you for calling Meridian Wireless, this is Tony. How can I help you today?
[00:06] CUSTOMER: This is the THIRD time I'm calling about the same problem. My bill is wrong EVERY month. I was told $75 but I keep getting charged $93.
[00:18] AGENT: I sincerely apologize for the ongoing frustration. Dealing with the same billing issue three times is completely unacceptable, and I understand why you're upset. Let me look at this very carefully and try to get to the bottom of it once and for all.
[00:30] CUSTOMER: Fine. AC-6681200. And I want a supervisor.
[00:35] AGENT: I hear you, and I'm happy to get you to a supervisor. Before I do, may I take a quick look so I can give them the full context? That way you won't have to re-explain everything.
[00:45] CUSTOMER: Make it quick.
[00:48] AGENT: I see it — there's an $18 international calling add-on that keeps reattaching after it's removed. It was removed twice before but came back each time. This looks like a system bug, not human error.
[01:05] CUSTOMER: A system bug?! And nobody caught this?
[01:10] AGENT: You're absolutely right to be frustrated. This should have been caught and escalated on your first call. I'm documenting everything now and transferring you to my supervisor Rachel, who can authorize a permanent fix and process refunds for the full $54 you've been overcharged. You won't need to re-explain anything.
[01:25] CUSTOMER: Alright. Just make sure it actually gets fixed this time.
[01:30] AGENT: I will. And I want you to know — your patience through this has been appreciated, even though we should have done better. Rachel will take great care of you. Transferring now.
""".strip(), {"team": "billing", "duration_sec": 90}),

    Conversation("conv_007", """
[00:00] AGENT: Hi welcome to Meridian
[00:04] CUSTOMER: Yeah hi, I want to add the international calling thing to my plan.
[00:10] AGENT: okay let me add that for you whats your account
[00:14] CUSTOMER: AC-5578231
[00:18] AGENT: ok I added international calling its $15 a month. its on your next bill
[00:25] CUSTOMER: Does that cover calls to Japan?
[00:28] AGENT: yeah it covers like 60 countries I think Japan is on there
[00:33] CUSTOMER: You think? Can you check?
[00:36] AGENT: yeah hold on... yeah Japan is included
[00:42] CUSTOMER: Okay great. Anything else I should know?
[00:45] AGENT: nope thats it. have a good day
""".strip(), {"team": "sales", "duration_sec": 45}),

    Conversation("conv_008", """
[00:00] AGENT: Thank you for calling Meridian Wireless, I'm Jessica. How can I assist you today?
[00:06] CUSTOMER: I need to set up parental controls on my kids' devices. They're spending way too much time online.
[00:14] AGENT: I totally understand that concern — it's tough to manage screen time. The good news is we have a free tool called Meridian Safe that gives you a lot of control. Have you heard of it?
[00:24] CUSTOMER: No, tell me about it.
[00:27] AGENT: So Meridian Safe is an app you download on your phone. Once you log in with your Meridian credentials, you can block website categories — like gaming or social media — set time limits per device, and even schedule bedtime shutoffs so the internet turns off automatically at a set time.
[00:42] CUSTOMER: Oh that's exactly what I need. Does it work on all their devices?
[00:47] AGENT: Yes, it works on anything connected to your home WiFi — phones, tablets, gaming consoles, everything. The one thing to know is it only works on your home network, so if they switch to cellular data, the controls won't apply there.
[01:00] CUSTOMER: That makes sense. How do I set it up?
[01:04] AGENT: Just search for Meridian Safe in your app store, download it, and log in with the same credentials you use for your Meridian account. The setup wizard walks you through everything. It takes about 5 minutes.
[01:15] CUSTOMER: Wonderful, I'll do that tonight.
[01:18] AGENT: I'm glad I could help. And if you run into any issues during setup, don't hesitate to call us back. Is there anything else?
[01:25] CUSTOMER: Nope, that's everything. Thank you Jessica!
[01:28] AGENT: You're welcome! Have a great evening.
""".strip(), {"team": "general", "duration_sec": 88}),
]

# ─── Human rater scores (2 raters per conversation) ───────────────────────

HUMAN_SCORES = [
    # conv_001 — clean billing resolution
    HumanScore("conv_001", "rater_A", {"greeting": 5, "problem_solving": 5, "empathy": 4, "resolution": 5, "closing": 5}),
    HumanScore("conv_001", "rater_B", {"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5}),

    # conv_002 — brusque tech escalation
    HumanScore("conv_002", "rater_A", {"greeting": 1, "problem_solving": 3, "empathy": 1, "resolution": 3, "closing": 1}),
    HumanScore("conv_002", "rater_B", {"greeting": 2, "problem_solving": 3, "empathy": 2, "resolution": 2, "closing": 2}),

    # conv_003 — excellent retention save
    HumanScore("conv_003", "rater_A", {"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5}),
    HumanScore("conv_003", "rater_B", {"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5}),

    # conv_004 — solid multi-issue handling
    HumanScore("conv_004", "rater_A", {"greeting": 4, "problem_solving": 5, "empathy": 3, "resolution": 5, "closing": 5}),
    HumanScore("conv_004", "rater_B", {"greeting": 4, "problem_solving": 4, "empathy": 3, "resolution": 4, "closing": 4}),

    # conv_005 — short, minimal effort
    HumanScore("conv_005", "rater_A", {"greeting": 2, "problem_solving": 4, "empathy": 2, "resolution": 4, "closing": 2}),
    HumanScore("conv_005", "rater_B", {"greeting": 3, "problem_solving": 4, "empathy": 1, "resolution": 4, "closing": 3}),

    # conv_006 — excellent handling of angry repeat caller
    HumanScore("conv_006", "rater_A", {"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 4, "closing": 5}),
    HumanScore("conv_006", "rater_B", {"greeting": 5, "problem_solving": 4, "empathy": 5, "resolution": 5, "closing": 5}),

    # conv_007 — casual, unprofessional, task completed
    HumanScore("conv_007", "rater_A", {"greeting": 2, "problem_solving": 3, "empathy": 1, "resolution": 3, "closing": 1}),
    HumanScore("conv_007", "rater_B", {"greeting": 1, "problem_solving": 4, "empathy": 1, "resolution": 4, "closing": 2}),

    # conv_008 — excellent customer education
    HumanScore("conv_008", "rater_A", {"greeting": 5, "problem_solving": 5, "empathy": 4, "resolution": 5, "closing": 5}),
    HumanScore("conv_008", "rater_B", {"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5}),
]


# ─── Mock LLM scorer (do NOT modify) ─────────────────────────────────────

def mock_llm_judge(prompt: str) -> str:
    """
    Simulates an LLM scoring a conversation. Returns JSON with scores
    and brief justifications. Has intentional biases for Part 4 analysis:
    - Tends to be lenient on empathy (scores 1 point higher than humans)
    - Tends to be harsh on short conversations
    - Occasionally disagrees on resolution for escalated calls
    """
    if "conv_001" in prompt:
        return json.dumps({"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5,
            "justifications": {"greeting": "Professional greeting with name", "problem_solving": "Quickly identified system migration issue", "empathy": "Acknowledged frustration and apologized", "resolution": "Full refund issued and preventive note added", "closing": "Confirmed actions and warm close"}})
    elif "conv_002" in prompt:
        return json.dumps({"greeting": 1, "problem_solving": 3, "empathy": 3, "resolution": 3, "closing": 1,
            "justifications": {"greeting": "No name, no greeting", "problem_solving": "Identified the issue correctly", "empathy": "Said 'I understand' briefly", "resolution": "Escalated appropriately but didn't offer interim help", "closing": "Abrupt, no summary"}})
    elif "conv_003" in prompt:
        return json.dumps({"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5,
            "justifications": {"greeting": "Warm, personal greeting", "problem_solving": "Identified pricing concern and found matching offer", "empathy": "Validated feelings about overpaying, acknowledged loyalty", "resolution": "Clear offer with transparent terms", "closing": "Confirmed changes, sent email, warm close"}})
    elif "conv_004" in prompt:
        return json.dumps({"greeting": 4, "problem_solving": 5, "empathy": 4, "resolution": 5, "closing": 5,
            "justifications": {"greeting": "Professional with name", "problem_solving": "Handled two issues efficiently", "empathy": "Showed understanding of inconvenience", "resolution": "Clear actions for both issues with reference numbers", "closing": "Excellent recap of all actions taken"}})
    elif "conv_005" in prompt:
        return json.dumps({"greeting": 1, "problem_solving": 3, "empathy": 1, "resolution": 3, "closing": 1,
            "justifications": {"greeting": "Bare minimum, no name", "problem_solving": "Answered the question but nothing more", "empathy": "No acknowledgment of customer needs", "resolution": "Correct answer but no proactive info", "closing": "Abrupt, no follow-up offer"}})
    elif "conv_006" in prompt:
        return json.dumps({"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 3, "closing": 5,
            "justifications": {"greeting": "Professional and warm", "problem_solving": "Identified root cause system bug that others missed", "empathy": "Exceptional — validated frustration, took ownership of past failures", "resolution": "Transferred to supervisor but didn't resolve directly", "closing": "Documented everything, smooth transfer, acknowledged patience"}})
    elif "conv_007" in prompt:
        return json.dumps({"greeting": 2, "problem_solving": 4, "empathy": 2, "resolution": 4, "closing": 2,
            "justifications": {"greeting": "Informal but present", "problem_solving": "Added the service, confirmed Japan coverage", "empathy": "No acknowledgment of customer needs", "resolution": "Task completed correctly", "closing": "Brief, no summary or follow-up"}})
    elif "conv_008" in prompt:
        return json.dumps({"greeting": 5, "problem_solving": 5, "empathy": 5, "resolution": 5, "closing": 5,
            "justifications": {"greeting": "Professional, friendly, with name", "problem_solving": "Thorough walkthrough of Meridian Safe features", "empathy": "Acknowledged screen time challenge as a parent", "resolution": "Clear setup instructions with realistic time estimate", "closing": "Offered follow-up support, warm close"}})
    else:
        return json.dumps({"greeting": 3, "problem_solving": 3, "empathy": 3, "resolution": 3, "closing": 3,
            "justifications": {c: "Unable to evaluate" for c in CRITERIA}})


# ─────────────────────────────────────────────────────────────────────────────
# PART 1: SCORING PROMPT & RUBRIC SYSTEM                          [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Build the prompt that turns an LLM into a conversation quality judge.
# ─────────────────────────────────────────────────────────────────────────────

def build_scoring_prompt(conversation: Conversation) -> str:
    """
    Build a prompt that asks the LLM to score a conversation on the rubric.

    Requirements:
      - Define each criterion with clear scoring anchors (what does 1 vs 5 look like?)
      - Include the conversation transcript
      - Request scores as JSON with justifications per criterion
      - Instruct the model to score based ONLY on what's in the transcript
      - Format: {"greeting": int, "problem_solving": int, ..., "justifications": {...}}

    THINK ABOUT:
      - Scoring anchors matter a lot. "Good empathy" is vague. "Agent explicitly
        acknowledged the customer's emotion using specific language" is testable.
      - Should you include examples of 1-score vs 5-score conversations?
      - How do you prevent the LLM from defaulting to 3 on everything?
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1: Implement build_scoring_prompt")


def parse_judge_response(raw: str) -> Tuple[Optional[Dict[str, int]], Optional[Dict[str, str]], List[str]]:
    """
    Parse the LLM judge's response into scores and justifications.

    Returns: (scores, justifications, errors)
      - scores: {"greeting": 4, "problem_solving": 5, ...} or None
      - justifications: {"greeting": "...", ...} or None
      - errors: list of validation issues

    Validate:
      - All 5 criteria present
      - All scores are integers 1-5
      - Justifications are non-empty strings
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1b: Implement parse_judge_response")


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: RUN THE LLM JUDGE                                       [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────

def score_conversation(conversation: Conversation) -> Dict:
    """
    Score a single conversation using the LLM judge.

    Returns:
    {
        "conversation_id": str,
        "scores": {"greeting": int, ...},
        "justifications": {"greeting": str, ...},
        "total_score": int,     # sum of all 5 criteria (out of 25)
        "avg_score": float,     # average across criteria
        "errors": [str],
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2a: Implement score_conversation")


def score_all_conversations(conversations: List[Conversation]) -> List[Dict]:
    """Score every conversation. Return list of results from score_conversation."""
    # TODO: Implement this
    raise NotImplementedError("Part 2b: Implement score_all_conversations")


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: CALIBRATE AGAINST HUMAN RATERS                          [20 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# The LLM judge is useless if it doesn't agree with human QA raters.
# Measure agreement and identify where the LLM diverges.
# ─────────────────────────────────────────────────────────────────────────────

def get_human_consensus(human_scores: List[HumanScore]) -> Dict[str, Dict[str, float]]:
    """
    Compute the average human score per conversation per criterion.

    Returns:
    {
        "conv_001": {"greeting": 5.0, "problem_solving": 5.0, ...},
        "conv_002": {"greeting": 1.5, "problem_solving": 3.0, ...},
        ...
    }

    Why average? With only 2 raters, the mean is our best proxy for
    "true" quality. In production with 3+ raters, you'd use median.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3a: Implement get_human_consensus")


def compute_agreement(
    llm_results: List[Dict],
    human_consensus: Dict[str, Dict[str, float]],
) -> Dict:
    """
    Compare LLM scores vs human consensus.

    Returns:
    {
        "per_criterion": {
            "greeting": {
                "mae": float,       # mean absolute error (LLM vs human avg)
                "bias": float,      # mean signed error (positive = LLM higher)
                "exact_match": float, # % where LLM == round(human_avg)
            },
            ...
        },
        "overall_mae": float,
        "overall_bias": float,
        "overall_exact_match": float,
    }

    MAE (Mean Absolute Error): average of |llm_score - human_avg| across all
    conversations. Lower is better. An MAE of 0.5 means the LLM is off by
    half a point on average.

    Bias: average of (llm_score - human_avg). Positive means the LLM tends
    to score higher than humans (lenient). Negative means harsher.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3b: Implement compute_agreement")


def cohens_kappa(
    llm_results: List[Dict],
    human_consensus: Dict[str, Dict[str, float]],
) -> Dict:
    """
    Compute Cohen's Kappa between LLM and human consensus (rounded).

    Kappa measures agreement beyond chance:
      kappa = (observed_agreement - chance_agreement) / (1 - chance_agreement)

    For this problem, simplify to:
      - Round human consensus to nearest int
      - Compare LLM int scores vs rounded human scores
      - Compute kappa over ALL (conversation, criterion) pairs

    Interpretation:
      < 0.20: poor
      0.21-0.40: fair
      0.41-0.60: moderate
      0.61-0.80: substantial
      0.81-1.00: near-perfect

    Returns:
    {
        "kappa": float,
        "observed_agreement": float,  # % of exact matches
        "chance_agreement": float,    # expected by random
        "interpretation": str,
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3c: Implement cohens_kappa")


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: DISAGREEMENT ANALYSIS & BIAS                            [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────

def find_disagreements(
    llm_results: List[Dict],
    human_consensus: Dict[str, Dict[str, float]],
    threshold: float = 1.5,
) -> List[Dict]:
    """
    Find cases where LLM and humans disagree by more than threshold points.

    Returns list sorted by disagreement magnitude (largest first):
    [
        {
            "conversation_id": str,
            "criterion": str,
            "llm_score": int,
            "human_avg": float,
            "difference": float,  # llm - human (positive = LLM higher)
            "llm_justification": str,
        }
    ]
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4a: Implement find_disagreements")


def analyze_bias(
    llm_results: List[Dict],
    human_consensus: Dict[str, Dict[str, float]],
    conversations: List[Conversation],
) -> Dict:
    """
    Analyze systematic biases in the LLM judge.

    Check for:
    1. Per-criterion bias: does the LLM consistently over/under-score
       certain criteria?
    2. Length bias: does the LLM score short conversations lower?
    3. Score-level bias: is the LLM more accurate on good vs bad agents?

    Returns:
    {
        "criterion_bias": {
            "greeting": float,    # avg (llm - human) for this criterion
            "empathy": float,     # positive = LLM is more lenient
            ...
        },
        "length_bias": {
            "short_calls_avg_diff": float,   # calls < 60 sec
            "long_calls_avg_diff": float,    # calls >= 60 sec
        },
        "score_level_bias": {
            "low_scoring_diff": float,   # for conversations where human avg < 3
            "high_scoring_diff": float,  # for conversations where human avg >= 4
        },
        "summary": str,  # 2-3 sentences describing the most important bias
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4b: Implement analyze_bias")


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("CRESTA MLE — LLM-as-Judge Auto-Scorer")
    print("=" * 60)

    # ── Part 1 ──────────────────────────────────────────────────────
    print("\n📝 PART 1: Scoring Prompt")
    print("-" * 40)
    try:
        prompt = build_scoring_prompt(CONVERSATIONS[0])
        print(f"  Prompt length: {len(prompt.split())} words")
        print(f"  Preview: \"{prompt[:100]}...\"")
        print("  ✅ Part 1a passed")
    except NotImplementedError:
        print("  ⬜ Part 1a: Not implemented yet")

    try:
        raw = '{"greeting": 5, "problem_solving": 4, "empathy": 3, "resolution": 5, "closing": 4, "justifications": {"greeting": "good", "problem_solving": "fine", "empathy": "ok", "resolution": "clear", "closing": "nice"}}'
        scores, justifications, errors = parse_judge_response(raw)
        assert scores is not None and scores["greeting"] == 5
        print(f"  Parser test: OK, {len(errors)} errors")
        print("  ✅ Part 1b passed")
    except NotImplementedError:
        print("  ⬜ Part 1b: Not implemented yet")

    # ── Part 2 ──────────────────────────────────────────────────────
    print(f"\n🤖 PART 2: LLM Judge Scoring")
    print("-" * 40)
    llm_results = []
    try:
        llm_results = score_all_conversations(CONVERSATIONS)
        for r in llm_results:
            print(f"  {r['conversation_id']}: total={r['total_score']}/25 avg={r['avg_score']:.1f}")
        print("  ✅ Part 2 passed")
    except NotImplementedError:
        print("  ⬜ Part 2: Not implemented yet")

    # ── Part 3 ──────────────────────────────────────────────────────
    print(f"\n📊 PART 3: Calibration vs Humans")
    print("-" * 40)
    try:
        consensus = get_human_consensus(HUMAN_SCORES)
        print(f"  Human consensus for {len(consensus)} conversations:")
        for cid, scores in sorted(consensus.items()):
            avg = sum(scores.values()) / len(scores)
            print(f"    {cid}: avg={avg:.1f}")
        print("  ✅ Part 3a passed")
    except NotImplementedError:
        print("  ⬜ Part 3a: Not implemented yet")

    try:
        if llm_results:
            agreement = compute_agreement(llm_results, consensus)
            print(f"\n  Agreement metrics:")
            print(f"    Overall MAE:         {agreement['overall_mae']:.3f}")
            print(f"    Overall bias:        {agreement['overall_bias']:+.3f}")
            print(f"    Overall exact match: {agreement['overall_exact_match']:.1%}")
            print(f"  Per criterion:")
            for c in CRITERIA:
                m = agreement["per_criterion"][c]
                print(f"    {c:<18s} MAE={m['mae']:.2f}  bias={m['bias']:+.2f}  exact={m['exact_match']:.0%}")
            print("  ✅ Part 3b passed")
    except NotImplementedError:
        print("  ⬜ Part 3b: Not implemented yet")

    try:
        if llm_results:
            kappa = cohens_kappa(llm_results, consensus)
            print(f"\n  Cohen's Kappa: {kappa['kappa']:.3f} ({kappa['interpretation']})")
            print(f"    Observed agreement: {kappa['observed_agreement']:.1%}")
            print(f"    Chance agreement:   {kappa['chance_agreement']:.1%}")
            print("  ✅ Part 3c passed")
    except NotImplementedError:
        print("  ⬜ Part 3c: Not implemented yet")

    # ── Part 4 ──────────────────────────────────────────────────────
    print(f"\n🔍 PART 4: Disagreement Analysis")
    print("-" * 40)
    try:
        if llm_results:
            disagreements = find_disagreements(llm_results, consensus)
            print(f"  Found {len(disagreements)} major disagreements:")
            for d in disagreements[:5]:
                direction = "LLM higher" if d["difference"] > 0 else "LLM lower"
                print(f"    {d['conversation_id']}/{d['criterion']}: "
                      f"LLM={d['llm_score']} human={d['human_avg']:.1f} ({direction})")
            print("  ✅ Part 4a passed")
    except NotImplementedError:
        print("  ⬜ Part 4a: Not implemented yet")

    try:
        if llm_results:
            bias = analyze_bias(llm_results, consensus, CONVERSATIONS)
            print(f"\n  Bias analysis:")
            print(f"    Criterion biases: ", end="")
            for c in CRITERIA:
                print(f"{c}={bias['criterion_bias'][c]:+.2f} ", end="")
            print()
            print(f"    Short vs long calls: short={bias['length_bias']['short_calls_avg_diff']:+.2f}, long={bias['length_bias']['long_calls_avg_diff']:+.2f}")
            print(f"    Low vs high scoring: low={bias['score_level_bias']['low_scoring_diff']:+.2f}, high={bias['score_level_bias']['high_scoring_diff']:+.2f}")
            print(f"    Summary: {bias['summary']}")
            print("  ✅ Part 4b passed")
    except NotImplementedError:
        print("  ⬜ Part 4b: Not implemented yet")

    print("\n" + "=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
================================================================================
SIMPLE SOLUTION — Cresta LLM-as-Judge Auto-Scorer (realistic for 1 hour)
================================================================================
"""

import json
from typing import Dict, List, Tuple, Optional
from collections import Counter, defaultdict

from cresta_judge_problem import (
    Conversation, HumanScore, CRITERIA,
    CONVERSATIONS, HUMAN_SCORES,
    mock_llm_judge,
)


# ═══════════════════════════════════════════════════════════════════════
# PART 1 — build scoring prompt + parse response
# ═══════════════════════════════════════════════════════════════════════

def build_scoring_prompt(conversation: Conversation) -> str:
    return f"""You are a QA scorer for a contact center. Score this agent conversation on 5 criteria, each 1-5.

SCORING RUBRIC:
1. greeting (1-5):
   1 = No greeting or name. 3 = Greeting present but generic. 5 = Warm greeting with agent's name and offer to help.
2. problem_solving (1-5):
   1 = Did not address the issue. 3 = Addressed but incompletely. 5 = Fully understood and resolved efficiently.
3. empathy (1-5):
   1 = No acknowledgment of customer feelings. 3 = Brief "I understand." 5 = Specifically named the emotion and validated it.
4. resolution (1-5):
   1 = No resolution or next step. 3 = Partial resolution. 5 = Clear resolution with specific actions and confirmation.
5. closing (1-5):
   1 = Abrupt end. 3 = Basic goodbye. 5 = Summarized actions, offered further help, warm close.

CONVERSATION ({conversation.id}):
{conversation.transcript}

Score ONLY what you see in the transcript. Output JSON:
{{"greeting": int, "problem_solving": int, "empathy": int, "resolution": int, "closing": int, "justifications": {{"greeting": "...", ...}}}}"""


def parse_judge_response(raw: str) -> Tuple[Optional[Dict[str, int]], Optional[Dict[str, str]], List[str]]:
    errors = []

    try:
        parsed = json.loads(raw.strip())
    except json.JSONDecodeError as e:
        return None, None, [f"JSON parse error: {e}"]

    scores = {}
    justifications = {}

    for c in CRITERIA:
        if c not in parsed:
            errors.append(f"Missing criterion: {c}")
            continue
        val = parsed[c]
        if not isinstance(val, int) or val < 1 or val > 5:
            errors.append(f"{c}: score {val} not in range 1-5")
            continue
        scores[c] = val

    if "justifications" in parsed and isinstance(parsed["justifications"], dict):
        justifications = parsed["justifications"]

    if len(scores) < 5:
        return None, None, errors

    return scores, justifications, errors


# ═══════════════════════════════════════════════════════════════════════
# PART 2 — run LLM judge
# ═══════════════════════════════════════════════════════════════════════

def score_conversation(conversation: Conversation) -> Dict:
    prompt = build_scoring_prompt(conversation)
    raw = mock_llm_judge(prompt)
    scores, justifications, errors = parse_judge_response(raw)

    if scores is None:
        scores = {c: 0 for c in CRITERIA}
        justifications = {}

    total = sum(scores.values())
    avg = total / len(scores)

    return {
        "conversation_id": conversation.id,
        "scores": scores,
        "justifications": justifications or {},
        "total_score": total,
        "avg_score": round(avg, 1),
        "errors": errors,
    }


def score_all_conversations(conversations: List[Conversation]) -> List[Dict]:
    return [score_conversation(c) for c in conversations]


# ═══════════════════════════════════════════════════════════════════════
# PART 3 — calibrate against human raters
# ═══════════════════════════════════════════════════════════════════════

def get_human_consensus(human_scores: List[HumanScore]) -> Dict[str, Dict[str, float]]:
    """Average the two raters' scores per conversation per criterion."""
    grouped = defaultdict(list)
    for hs in human_scores:
        grouped[hs.conversation_id].append(hs.scores)

    consensus = {}
    for cid, score_list in grouped.items():
        consensus[cid] = {}
        for c in CRITERIA:
            vals = [s[c] for s in score_list if c in s]
            consensus[cid][c] = sum(vals) / len(vals) if vals else 0
    return consensus


def compute_agreement(llm_results, human_consensus):
    per_criterion = {c: {"errors": [], "signed_errors": [], "exact": []} for c in CRITERIA}

    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        for c in CRITERIA:
            llm = r["scores"][c]
            human = human_consensus[cid][c]
            diff = llm - human
            per_criterion[c]["errors"].append(abs(diff))
            per_criterion[c]["signed_errors"].append(diff)
            per_criterion[c]["exact"].append(1 if llm == round(human) else 0)

    result = {"per_criterion": {}}
    all_errors, all_signed, all_exact = [], [], []

    for c in CRITERIA:
        errs = per_criterion[c]["errors"]
        signed = per_criterion[c]["signed_errors"]
        exact = per_criterion[c]["exact"]
        result["per_criterion"][c] = {
            "mae": sum(errs) / max(len(errs), 1),
            "bias": sum(signed) / max(len(signed), 1),
            "exact_match": sum(exact) / max(len(exact), 1),
        }
        all_errors.extend(errs)
        all_signed.extend(signed)
        all_exact.extend(exact)

    result["overall_mae"] = sum(all_errors) / max(len(all_errors), 1)
    result["overall_bias"] = sum(all_signed) / max(len(all_signed), 1)
    result["overall_exact_match"] = sum(all_exact) / max(len(all_exact), 1)
    return result


def cohens_kappa(llm_results, human_consensus):
    """
    Cohen's kappa comparing LLM scores vs rounded human consensus.
    Computed over all (conversation, criterion) pairs.
    """
    llm_labels = []
    human_labels = []

    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        for c in CRITERIA:
            llm_labels.append(r["scores"][c])
            human_labels.append(round(human_consensus[cid][c]))

    n = len(llm_labels)
    if n == 0:
        return {"kappa": 0, "observed_agreement": 0, "chance_agreement": 0, "interpretation": "no data"}

    # Observed agreement: fraction of exact matches
    matches = sum(1 for a, b in zip(llm_labels, human_labels) if a == b)
    observed = matches / n

    # Chance agreement: probability both raters pick the same label by chance
    # P(both pick k) = P(llm picks k) * P(human picks k)
    categories = set(llm_labels) | set(human_labels)
    llm_counts = Counter(llm_labels)
    human_counts = Counter(human_labels)
    chance = sum((llm_counts[k] / n) * (human_counts[k] / n) for k in categories)

    if chance >= 1.0:
        kappa = 1.0
    else:
        kappa = (observed - chance) / (1 - chance)

    # Interpretation
    if kappa > 0.80:
        interp = "near-perfect"
    elif kappa > 0.60:
        interp = "substantial"
    elif kappa > 0.40:
        interp = "moderate"
    elif kappa > 0.20:
        interp = "fair"
    else:
        interp = "poor"

    return {
        "kappa": round(kappa, 3),
        "observed_agreement": round(observed, 3),
        "chance_agreement": round(chance, 3),
        "interpretation": interp,
    }


# ═══════════════════════════════════════════════════════════════════════
# PART 4 — disagreements and bias
# ═══════════════════════════════════════════════════════════════════════

def find_disagreements(llm_results, human_consensus, threshold=1.5):
    disagreements = []

    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        for c in CRITERIA:
            llm = r["scores"][c]
            human = human_consensus[cid][c]
            diff = llm - human
            if abs(diff) >= threshold:
                disagreements.append({
                    "conversation_id": cid,
                    "criterion": c,
                    "llm_score": llm,
                    "human_avg": human,
                    "difference": diff,
                    "llm_justification": r.get("justifications", {}).get(c, ""),
                })

    disagreements.sort(key=lambda d: -abs(d["difference"]))
    return disagreements


def analyze_bias(llm_results, human_consensus, conversations):
    # 1. Per-criterion bias
    criterion_bias = {c: [] for c in CRITERIA}
    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        for c in CRITERIA:
            diff = r["scores"][c] - human_consensus[cid][c]
            criterion_bias[c].append(diff)

    criterion_bias_avg = {
        c: round(sum(diffs) / max(len(diffs), 1), 2)
        for c, diffs in criterion_bias.items()
    }

    # 2. Length bias: short (<60s) vs long (>=60s)
    conv_duration = {c.id: c.metadata.get("duration_sec", 0) for c in conversations}
    short_diffs, long_diffs = [], []

    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        duration = conv_duration.get(cid, 0)
        for c in CRITERIA:
            diff = r["scores"][c] - human_consensus[cid][c]
            if duration < 60:
                short_diffs.append(diff)
            else:
                long_diffs.append(diff)

    # 3. Score-level bias: low-scoring (<3 avg) vs high-scoring (>=4 avg)
    low_diffs, high_diffs = [], []
    for r in llm_results:
        cid = r["conversation_id"]
        if cid not in human_consensus:
            continue
        human_avg = sum(human_consensus[cid].values()) / len(CRITERIA)
        for c in CRITERIA:
            diff = r["scores"][c] - human_consensus[cid][c]
            if human_avg < 3:
                low_diffs.append(diff)
            elif human_avg >= 4:
                high_diffs.append(diff)

    def safe_avg(lst):
        return round(sum(lst) / max(len(lst), 1), 2)

    # Summary
    most_biased = max(CRITERIA, key=lambda c: abs(criterion_bias_avg[c]))
    most_biased_val = criterion_bias_avg[most_biased]
    direction = "lenient" if most_biased_val > 0 else "harsh"
    short_bias = safe_avg(short_diffs)
    summary = (
        f"The LLM is most biased on '{most_biased}' ({most_biased_val:+.2f}, {direction}). "
        f"It scores short calls {'lower' if short_bias < 0 else 'higher'} than humans ({short_bias:+.2f}). "
        f"Low-scoring conversations show a bias of {safe_avg(low_diffs):+.2f}, suggesting the LLM "
        f"{'inflates' if safe_avg(low_diffs) > 0 else 'deflates'} scores on weak calls."
    )

    return {
        "criterion_bias": criterion_bias_avg,
        "length_bias": {
            "short_calls_avg_diff": safe_avg(short_diffs),
            "long_calls_avg_diff": safe_avg(long_diffs),
        },
        "score_level_bias": {
            "low_scoring_diff": safe_avg(low_diffs),
            "high_scoring_diff": safe_avg(high_diffs),
        },
        "summary": summary,
    }


# ═══════════════════════════════════════════════════════════════════════
# RUNNER
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("SIMPLE SOLUTION — LLM-as-Judge Auto-Scorer")
    print("=" * 60)

    # Part 1
    prompt = build_scoring_prompt(CONVERSATIONS[0])
    print(f"\n📝 Part 1: Prompt = {len(prompt.split())} words")

    # Part 2
    print(f"\n🤖 Part 2: Scoring all conversations")
    llm_results = score_all_conversations(CONVERSATIONS)
    for r in llm_results:
        print(f"  {r['conversation_id']}: {r['total_score']}/25 (avg {r['avg_score']})")

    # Part 3
    print(f"\n📊 Part 3: Calibration")
    consensus = get_human_consensus(HUMAN_SCORES)
    agreement = compute_agreement(llm_results, consensus)
    print(f"  MAE: {agreement['overall_mae']:.3f}")
    print(f"  Bias: {agreement['overall_bias']:+.3f}")
    print(f"  Exact match: {agreement['overall_exact_match']:.1%}")
    print(f"  Per criterion:")
    for c in CRITERIA:
        m = agreement["per_criterion"][c]
        print(f"    {c:<18s} MAE={m['mae']:.2f}  bias={m['bias']:+.2f}  exact={m['exact_match']:.0%}")

    kappa_result = cohens_kappa(llm_results, consensus)
    print(f"\n  Cohen's Kappa: {kappa_result['kappa']:.3f} ({kappa_result['interpretation']})")

    # Part 4
    print(f"\n🔍 Part 4: Disagreements & Bias")
    disagreements = find_disagreements(llm_results, consensus)
    print(f"  {len(disagreements)} major disagreements:")
    for d in disagreements:
        dir_str = "LLM higher" if d["difference"] > 0 else "LLM lower"
        print(f"    {d['conversation_id']}/{d['criterion']}: LLM={d['llm_score']} human={d['human_avg']:.1f} ({dir_str})")
        print(f"      LLM says: \"{d['llm_justification'][:70]}\"")

    bias = analyze_bias(llm_results, consensus, CONVERSATIONS)
    print(f"\n  {bias['summary']}")

    print("\n" + "=" * 60)